In [1]:
print(123)

123


In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# Q1. First trace

In [23]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "rag.search",
    "context": {
        "trace_id": "0xf682259c72843d137ad0ff053a5fe18f",
        "span_id": "0x395a9b00da15f901",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x5a117816a5d838a4",
    "start_time": "2026-07-19T01:29:00.307003Z",
    "end_time": "2026-07-19T01:29:00.311274Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "search.query": "How does the agentic loop keep calling the model until it stops?",
        "search.num_results": 5,
        "search.result_count": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "98458756-218e-4286-874b-5edec7f2962a",
            "service.name": "course-assistant"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0xf682259c72843d137ad0ff053a5fe18f",
        "span_id": "0x55a76d64870da9f9",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x5a117816a5d838a4",
    "start_time": "2026-07-19T01:29:00.313497Z",
    "end_time": "2026-07-19T01:29:02.015660Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.request.model": "gpt-5.4-mini",
        "input_tokens": 7111,
        "output_tokens": 134
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "98458756-218e-4286-874b-5edec7f2962a",
            "service.name": "course-assistant"
        },
        "schema_url": ""
    }
}
{
    "name": "rag",
    "context": {
        "trace_id": "0xf682259c72843d137ad0ff

In [4]:
print(answer)

It keeps calling the model in a `while True` loop, then checks whether the response contains any `function_call` items.

- If there is a function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop.

So the stop condition is: **no function calls this turn**.


# Q2. Capturing metrics as span attributes

In [24]:
llm_span = {
    "name": "llm",
    "context": {
        "trace_id": "0xf682259c72843d137ad0ff053a5fe18f",
        "span_id": "0x55a76d64870da9f9",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x5a117816a5d838a4",
    "start_time": "2026-07-19T01:29:00.313497Z",
    "end_time": "2026-07-19T01:29:02.015660Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.request.model": "gpt-5.4-mini",
        "input_tokens": 7111,
        "output_tokens": 134
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "98458756-218e-4286-874b-5edec7f2962a",
            "service.name": "course-assistant"
        },
        "schema_url": ""
    }
}

In [25]:
llm_span['attributes']['input_tokens']

7111

# Q3. Span timing

In [26]:
from datetime import datetime

In [27]:
print(llm_span['end_time'])
print(llm_span['start_time'])

2026-07-19T01:29:02.015660Z
2026-07-19T01:29:00.313497Z


In [28]:
datetime.fromisoformat(llm_span["end_time"]) - datetime.fromisoformat(llm_span["start_time"])

datetime.timedelta(seconds=1, microseconds=702163)

# Q4. Saving traces to SQLite

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model with a `while True` loop. After each model response, it checks whether there were any `function_call` items:

- if there are function calls, it runs the tool, appends the tool result to `messages`, and calls the model again
- if there are no function calls, it breaks out of the loop

So the stop condition is: **no function calls this turn**.


In [4]:
import sqlite3

conn = sqlite3.connect("traces.db")
cur = conn.cursor()

In [5]:
cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
cur.fetchall()

[('spans',)]

In [12]:
import pandas as pd

pd.read_sql_query("SELECT * FROM spans LIMIT 20;", conn)

,name,start_time,end_time,input_tokens,output_tokens,cost
0,rag.search,1784425076523599973,1784425076525745491,NaN,NaN,None
1,llm,1784425076529055139,1784425078840113478,7111.0,110.0,None
2,rag,1784425076523479944,1784425078843521262,NaN,NaN,None
3,rag.search,1784425211409923307,1784425211414885721,NaN,NaN,None
4,llm,1784425211421535523,1784425213233684473,7111.0,91.0,None
5,rag,1784425211409861545,1784425213240903711,NaN,NaN,None


# Q5. Querying trace data

In [10]:
pd.read_sql_query("SELECT *, end_time - start_time FROM spans LIMIT 20;", conn)

,name,start_time,end_time,input_tokens,output_tokens,cost,end_time - start_time
0,rag.search,1784425076523599973,1784425076525745491,NaN,NaN,None,2145518
1,llm,1784425076529055139,1784425078840113478,7111.0,110.0,None,2311058339
2,rag,1784425076523479944,1784425078843521262,NaN,NaN,None,2320041318


# Q6. Token stability across runs

In [14]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model in a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there **is** a function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If there are **no** function calls, it `break`s out of the loop.

So the stop condition is: **no tool calls in the model’s response**.


In [15]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps looping with a `while True` block, and after each model call it checks whether the response contained any `function_call` items.

- If there was a function call, the code runs the tool, appends the tool output to `messages`, and continues.
- If there were no function calls, it `break`s out of the loop.

So the stop condition is: **no function calls in the model response**.


In [17]:
import pandas as pd

pd.read_sql_query("SELECT input_tokens FROM spans LIMIT 20;", conn)

,input_tokens
0,NaN
1,7111.0
2,NaN
3,NaN
4,7111.0
5,NaN
6,NaN
7,7111.0
8,NaN
9,NaN
